In [1]:
import pandas as pd
from matplotlib import pyplot as plt

# Checking the Data
This notebook examines the validity of the solutions found by the various tested algorithms by cross referencing the results and gathering information which may be used to construct a rigorous piece of code to confirm solutions to artificial game trees.
## Why is this difficult?
Artificial Game Trees, as generated according to previous research on B*, do not provide a way to easily check solutions. This is because the trees do not keep a predestined solution in mind while generating children bounds. While in most games we simply have to search for any winning position, for this solving task we are looking for the optimal move to maximize our score, regardless of which 'game-state' we end up in. The (P)B* algorithm can be adjusted to solve for this goal instead, but that is out of the scope of this research. Therefore, we must confirm that the move the algorithm found does indeed lead to the maximum score.
## Loading the data

In [2]:
df = pd.read_csv("../testing/alternate_runtest_001.csv"); df

,style,algorithm,growth,PS,num_alts,force_chance,seed,range,width,seconds,depth,evaluated,expanded,intractable,solution,depth1Pes,depth2Opt,sepValue
0,Berliner,prefer-DR,1.0,yes,2,0.0,6,100,3,0.035,2,10,3,no,3,1,1,63.0
1,Berliner,prefer-DR,1.0,yes,2,0.0,3,100,3,0.035,2,13,4,no,3,2,1,85.0
2,Berliner,prefer-DR,1.0,yes,2,0.0,5,100,3,0.035,6,28,9,no,2,1,1,49.0
3,Berliner,prefer-DR,1.0,yes,2,0.0,4,100,3,0.035,1,4,1,no,2,1,1,86.0
4,Berliner,prefer-DR,1.0,yes,2,0.0,8,100,3,0.035,2,10,3,no,1,1,1,58.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
679995,BaseRulesAfter,berlin-2R,1.4,no,5,1.0,67995,6400,9,0.023,26,30007,3334,yes,2,20,9,2897.0
679996,BaseRulesAfter,randomize,1.4,no,5,1.0,67998,6400,9,0.014,27,23941,2660,no,8,2,1,3438.0
679997,BaseRulesAfter,berlin-3R,1.4,no,5,1.0,67995,6400,9,0.023,26,30007,3334,yes,2,20,9,2954.0
679998,BaseRulesAfter,berlin-2R,1.4,no,5,1.0,67998,6400,9,0.022,31,30007,3334,yes,8,12,10,3252.0


In [3]:
per_tree = df[df.intractable == "no"].groupby('seed')
_num_solutions = per_tree.solution.apply(lambda x : len(x.unique()))
_num_sepvals = per_tree.sepValue.apply(lambda x : len(x.unique()))
df2 = df.join(pd.DataFrame({"numsol": _num_solutions, "numsep": _num_sepvals}), on="seed").fillna(0)

## Runs with different results from different algorithms

In [4]:
df2[df2.numsol > 1]

,style,algorithm,growth,PS,num_alts,force_chance,seed,range,width,seconds,depth,evaluated,expanded,intractable,solution,depth1Pes,depth2Opt,sepValue,numsol,numsep
1739,Berliner,prefer-DR,1.0,yes,2,0.0,178,100,5,0.0,7,216,43,no,2,2,1,66.0,2.0,1.0
1750,Berliner,always-PB,1.0,yes,2,0.0,178,100,5,0.0,7,101,20,no,5,2,1,66.0,2.0,1.0
1762,Berliner,berlin-2D,1.0,yes,2,0.0,178,100,5,0.0,7,136,27,no,5,7,3,66.0,2.0,1.0
1765,Berliner,alternate,1.0,yes,2,0.0,178,100,5,0.0,7,136,27,no,5,2,1,66.0,2.0,1.0
1769,Berliner,berlin-3D,1.0,yes,2,0.0,178,100,5,0.0,7,136,27,no,5,7,3,66.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660524,BaseRulesAfter,berlin-AR,1.4,no,2,1.0,66058,100,9,0.0,4,91,10,no,6,3,2,54.0,2.0,1.0
660525,BaseRulesAfter,berlin-AD,1.4,no,2,1.0,66058,100,9,0.0,4,91,10,no,6,3,2,54.0,2.0,1.0
660526,BaseRulesAfter,randomize,1.4,no,2,1.0,66058,100,9,0.0,3,64,7,no,6,2,1,54.0,2.0,1.0
660528,BaseRulesAfter,berlin-2R,1.4,no,2,1.0,66058,100,9,0.0,4,91,10,no,6,3,2,54.0,2.0,1.0


This seems worrying? Well: no! The number of solutions does not say everything, because a tree *could* have multiple solutions. If the solutions are all the same value, then it is still 100% correct. That is what 'sepValue' is for, to track the value at which separation occured during the search. If there are multiple solutions to the problem, then there must be multiple nodes with the same optimistic and pessimistic values $\to$ nodes with the same point-value $v_x$ such that $Opt(x)=Pes(x)=v_x$ (because node values always converge). In that case the solution will separate at $v_x$ when all (or all but one) of these multiple solutions are reduced to a point-value, which is $v_x$. That means: if all the different solutions have the same 'sepValue', then they are all valid (given that at least one of them is valid). Note that this only checks whether the different algorithms agree with one another; it does not check whether the values they got were actually correct.

In [5]:
df2[(df2.numsol > 1) & (df2.numsep > 1)]

,style,algorithm,growth,PS,num_alts,force_chance,seed,range,width,seconds,depth,evaluated,expanded,intractable,solution,depth1Pes,depth2Opt,sepValue,numsol,numsep


As seen above, there are $0$ examples of runs where there are multiple solutions that do not have the same value. So the algorithms agree on all cases that they could solve. Now is left to confirm that the solutions they found are correct.

## Confirming correctness of solutions
To confirm the correctness of the solutions we can really only do some kind of 'inherently correct' search which confirms the same answers we got from the B* algorithms. 